In [2]:
import pandas as pd
import win32com.client as win32
import os

# Instructivo:

#1) Crear una carpeta en el escritorio, con el nombre RG66

#2) Dentro de la carpeta tiene que estar el archivo de Excel "Documentos", en el cual se detallan los documentos ZP de la
#   transaccion FBL3N

#3) Abrir una cesion en SAP



#Detalla la direccion, en la cual se encuentra, el fichero desde el que estamos trabajando
fichero_actual = os.getcwd()



#Tomamos el año en formato string. Lo utilizaremos posteriormente en la funcion with_item
anio = input("ingrese el numero de año (yyyy):")


SapGuiAuto = win32.GetObject("SAPGUI")
application = SapGuiAuto.GetScriptingEngine
connection = application.Children(0)
session = connection.Children(0)



#Convierte a DF el cuadro de la hoja
documentos_excel = pd.read_excel("Documentos.xlsx")


#Eliminamos los indices, asi al ejecutar "to_clipboard()" no forman parte del numero de documento
documentos_excel = documentos_excel.set_index('Documentos')



################################
########### WITH_ITEM ##########
################################


session.findById("wnd[0]").maximize
session.findById("wnd[0]/tbar[0]/okcd").text = "/n"
session.findById("wnd[0]").sendVKey (0)
session.findById("wnd[0]/tbar[0]/okcd").text = "zse16"
session.findById("wnd[0]").sendVKey (0)
session.findById("wnd[0]/usr/ctxtP_TABLE").text = "with_item"
session.findById("wnd[0]/usr/ctxtP_TABLE").caretPosition = 9
session.findById("wnd[0]").sendVKey (0)
session.findById("wnd[0]/usr/ctxtI1-LOW").text = "a001"
session.findById("wnd[0]/usr/txtI3-LOW").text = anio
session.findById("wnd[0]/usr/txtMAX_SEL").text = "999999"
session.findById("wnd[0]/usr/txtI2-LOW").setFocus()
session.findById("wnd[0]/usr/txtI2-LOW").caretPosition = 0
session.findById("wnd[0]/usr/btn%_I2_%_APP_%-VALU_PUSH").press()


#Copia los documentos del dataframe al portapapeles
documentos_excel.to_clipboard()


session.findById("wnd[1]/tbar[0]/btn[24]").press()
session.findById("wnd[1]/tbar[0]/btn[8]").press()
session.findById("wnd[0]/tbar[1]/btn[8]").press()
session.findById("wnd[0]/tbar[1]/btn[45]").press()
session.findById("wnd[1]/usr/subSUBSCREEN_STEPLOOP:SAPLSPO5:0150/sub:SAPLSPO5:0150/radSPOPLI-SELFLAG[1,0]").select()
session.findById("wnd[1]/usr/subSUBSCREEN_STEPLOOP:SAPLSPO5:0150/sub:SAPLSPO5:0150/radSPOPLI-SELFLAG[1,0]").setFocus()
session.findById("wnd[1]/tbar[0]/btn[0]").press()
session.findById("wnd[1]/usr/ctxtDY_PATH").text = fichero_actual
session.findById("wnd[1]/usr/ctxtDY_FILENAME").text = "RG66.txt"
session.findById("wnd[1]/tbar[0]/btn[11]").press()



#Abre el archivo RG66 y selecciona las columnas a utilizar, asimismo les da el formato necesario para trabajarlas.
#Por ultimo las ordena
with_item = pd.read_csv("RG66.txt", sep="\t", encoding='latin1', 
                   usecols=["Soc.", "Nºcta.A/D", "Compens.", "Nº doc.", "Tipo retenciones", "Lib.mayor", "Ret", 
                            "Base imponible ret.", "Importe de retención", "ClCta", "Tp.reten.", "Nº certif."],
                   dtype={'Nºcta.A/D': 'Int64', 'Nº doc.': 'Int64', 'Lib.mayor': 'Int64', 'Ret': 'object',
                          'Nº certif.': 'object'},
                   skiprows=[0, 1, 2, 4], parse_dates=['Compens.'], dayfirst=True, thousands="."
                   , decimal=",")[["Soc.", "Nºcta.A/D", "Compens.", "Nº doc.", "Tipo retenciones", "Lib.mayor", "Ret", 
                            "Base imponible ret.", "Importe de retención", "ClCta", "Tp.reten.", "Nº certif."]]



#Filtra "with_item" por "IP" en la columna "Tipo retenciones"
with_item = with_item[with_item['Tipo retenciones'] == 'IP']


#Filtra "with_item" por "01" en la columna "Ret"
with_item = with_item[with_item['Ret'] == '01']


#Listamos los proveedores involucrados
lista_proveedores = with_item['Nºcta.A/D'].tolist()


#Renombramos la columna, para luego hacer merge
with_item = with_item.rename(columns={'Nºcta.A/D':'Proveedor'})



################################
############# LFA1 #############
################################



session.findById("wnd[0]").maximize
session.findById("wnd[0]/tbar[0]/okcd").text = "/n"
session.findById("wnd[0]").sendVKey (0)
session.findById("wnd[0]/tbar[0]/okcd").text = "zse16"
session.findById("wnd[0]").sendVKey (0)
session.findById("wnd[0]/usr/ctxtP_TABLE").text = "lfa1"
session.findById("wnd[0]/usr/ctxtP_TABLE").caretPosition = 4
session.findById("wnd[0]").sendVKey (0)
session.findById("wnd[0]/usr/btn%_I1_%_APP_%-VALU_PUSH").press()


#Tomamos la columna "Proveedor" como DataFrame y le quitamos el indice para que no interfiera con el numero de proveedor
proveedor = with_item[['Proveedor']].set_index('Proveedor')


#Copia los documentos del dataframe al portapapeles
proveedor.to_clipboard()


session.findById("wnd[1]/tbar[0]/btn[24]").press()
session.findById("wnd[1]/tbar[0]/btn[8]").press()
session.findById("wnd[0]/usr/txtMAX_SEL").text = "999999"
session.findById("wnd[0]/usr/txtMAX_SEL").setFocus()
session.findById("wnd[0]/usr/txtMAX_SEL").caretPosition = 11
session.findById("wnd[0]/tbar[1]/btn[8]").press()
session.findById("wnd[0]/tbar[1]/btn[45]").press()
session.findById("wnd[1]/usr/subSUBSCREEN_STEPLOOP:SAPLSPO5:0150/sub:SAPLSPO5:0150/radSPOPLI-SELFLAG[1,0]").select()
session.findById("wnd[1]/usr/subSUBSCREEN_STEPLOOP:SAPLSPO5:0150/sub:SAPLSPO5:0150/radSPOPLI-SELFLAG[1,0]").setFocus()
session.findById("wnd[1]/tbar[0]/btn[0]").press()
session.findById("wnd[1]/usr/ctxtDY_PATH").text = fichero_actual
session.findById("wnd[1]/usr/ctxtDY_FILENAME").text = "LFA1.txt"
session.findById("wnd[1]/tbar[0]/btn[11]").press()



lfa1 = pd.read_csv("LFA1.txt", sep="\t", encoding='latin1', 
                   usecols=["Proveedor", "Nombre 1", "Nº ident.fis.1"],
                   dtype={'Proveedor': 'Int64', 'Nº ident.fis.1': 'Int64'},
                   skiprows=[0, 1, 2, 4])[['Proveedor', 'Nombre 1', 'Nº ident.fis.1']]


#Borra todas las filas que contengan NaN en cada una de las columnas
lfa1 = lfa1.dropna(how='all')



rg66 = pd.merge(with_item, lfa1, how = 'left', on = "Proveedor")


#Ordena los nombres de las columnas
rg66 = rg66[['Soc.', 'Proveedor', 'Nombre 1', 'Nº ident.fis.1', 'Compens.', 'Nº doc.', 'Tipo retenciones',
                     'Lib.mayor', 'Ret', 'Base imponible ret.', 'Importe de retención', 'ClCta',
                     'Tp.reten.', 'Nº certif.']]


#Totaliza la columna 'Importe de retención'
total = rg66['Importe de retención'].sum()


#Obtenemos la cantidad de filas del DataFrame y le añadimos 3 para posicionar el total
posi = str(rg66.shape[0] + 4)



################################
########## XLSXWRITER ##########
################################


#El formato de fecha debemos configurarlo en este punto, ya que Xlsxwriter no permite hacerlo desde set_column
with pd.ExcelWriter("RG66.xlsx", datetime_format='dd/mm/yyyy') as writer:
    rg66.to_excel(writer, index = False, sheet_name = "Consolidado", startrow = 2, startcol = 1)
    
    
    
    workbook = writer.book

    worksheet = writer.sheets["Consolidado"]
    #Now we have the worksheet object. We can manipulate it
    
   
    #_Formato numero con 2 decimales y separador de miles    
    header_format_4 = workbook.add_format({
            'num_format': '#,##0.00'})
    
    
    #_Color de las celdas del titulo   
    header_format_0 = workbook.add_format({
        "border": 1, #borde
        "border_color": "#000000", #color de borde            
        "valign": "vcenter", #centra el texto en vertical
        "align": "center", #centra el texto en horizontal        
        "bg_color": "#4F81BD",
        "font_color": "#FFFFFF", #color de la fuente
        "font_size": 11,
        "font_name": "Calibri"})
    
    
    
    #_Color de las celdas del titulo   
    header_format_1 = workbook.add_format({
        "border": 1, #borde
        "border_color": "#000000", #color de borde            
        "valign": "vcenter", #centra el texto en vertical
        "align": "center", #centra el texto en horizontal        
        "bg_color": "#FFEB9C",
        "font_color": "#9C5700", #color de la fuente
        "font_size": 11,
        "font_name": "Calibri"})
    
    
    
    #_Color de las celdas del titulo   
    header_format_2 = workbook.add_format({
        "border": 2, #borde
        "border_color": "#000000",
        'num_format': '#,##0.00'})
    
    
        
    #_Color de las celdas a utilizar en contorno vertical 
    header_format_3 = workbook.add_format({
        "right": 2,
        "right_color": "#000000"})
    
    
    
    #_Color de las celdas a utilizar en contorno horizontal 
    header_format_5 = workbook.add_format({
        "top": 2,
        "top_color": "#000000"})
    
    
    
    #_Formato numero sin decimales    
    header_format_6 = workbook.add_format({
            'num_format': '##0'})    

        
    
    #_Ancho de las columnas y formato
    worksheet.set_column('D:D', 32)
    worksheet.set_column('E:E', 12.57)
    worksheet.set_column('E:E', 12.57)
    worksheet.set_column('F:F', 12)
    worksheet.set_column('G:G', 12)
    worksheet.set_column('H:H', 15.14)
    worksheet.set_column('I:I', 11.43)
    worksheet.set_column('K:K', 17.71, header_format_6)
    worksheet.set_column('L:L', 18, header_format_4)
    worksheet.set_column('N:N', 8.43, header_format_4)
    worksheet.set_column('O:O', 11.43)
    
    
    
    #Renombra las columnas y les asigna formato
    worksheet.write('B3', 'Soc.', header_format_0)
    worksheet.write('C3', 'Proveedor', header_format_0)
    worksheet.write('D3', 'Razon Social', header_format_0)
    worksheet.write('E3', 'Cuit', header_format_1)
    worksheet.write('F3', 'Fecha', header_format_0)
    worksheet.write('G3', 'Nº doc.', header_format_0)
    worksheet.write('H3', 'Tipo retenciones', header_format_0)
    worksheet.write('I3', 'Lib.mayor', header_format_0)
    worksheet.write('J3', 'Ret', header_format_0)
    worksheet.write('K3', 'Base imponible ret.', header_format_1)
    worksheet.write('L3', 'Importe de retención', header_format_0)
    worksheet.write('M3', 'ClCta', header_format_0)
    worksheet.write('N3', 'Tp.reten.', header_format_0)
    worksheet.write('O3', 'Nº certif.', header_format_0)
    worksheet.write('L' + posi, total, header_format_2)
    
    
    
    #Contorno negro del cuadro general
    #Definimos los lados del rango del cuadro
    rango_cuadro1 = "A4:A" + str(rg66.shape[0] + 3)
    rango_cuadro2 = "O4:O" + str(rg66.shape[0] + 3)
    lados = rango_cuadro1 + " " + rango_cuadro2
    
    
    
    #Se utiliza el formato condicional, ya es el que nos permite aplicar el formato sobre un rango especifico
    #Se aplica sobre el cuadro
    worksheet.conditional_format(rango_cuadro1, {'type' : 'formula',
                                 'criteria' : '=$B$4="A001"', 
                                 'format': header_format_3,
                                 'multi_range': lados}) #Extiende el rango sobre el cual aplica el formato

    
    
    #Contorno negro del cuadro general
    #Definimos el rango del contorno horizontal del cuadro
    rango_cuadro3 = "B4:O4"
    rango_cuadro4 = "B" + str(rg66.shape[0] + 4) + ":" + "O" + str(rg66.shape[0] + 4)
    horizontal = rango_cuadro3 + " " + rango_cuadro4
    
    
    
    #Se utiliza el formato condicional, ya es el que nos permite aplicar el formato sobre un rango especifico
    #Se aplica sobre el cuadro
    worksheet.conditional_format(rango_cuadro3, {'type' : 'formula',
                                 'criteria' : '=$B$4="A001"', #Si se cumple la condicion
                                 'format': header_format_5,
                                 'multi_range': horizontal}) #Extiende el rango sobre el cual aplica el formato
    
    
    
    #Oculta las lineas de cuadricula de la hoja
    worksheet.hide_gridlines([1])


    #Porcentaje visualizacion de la hoja
    worksheet.set_zoom(90)

ingrese el numero de año (yyyy):2024
